# Day 9: Memory — RAM, ROM & Block RAM

## Pre-Class Videos (~45 minutes total)

| # | Segment | Duration | File |
|---|---------|----------|------|
| 1 | ROM in Verilog | ~12 min | `d09_s1_rom_in_verilog.html` |
| 2 | RAM in Verilog | ~12 min | `d09_s2_ram_in_verilog.html` |
| 3 | iCE40 Memory Resources | ~10 min | `d09_s3_ice40_memory_resources.html` |
| 4 | Practical Memory Applications | ~11 min | `d09_s4_memory_applications.html` |

## Code Examples

| File | Description |
|------|-------------|
| `code/day09_ex01_rom_sync.v` | Parameterized synchronous ROM with `$readmemb`, self-checking TB |
| `code/day09_ex02_ram_sp.v` | Single-port synchronous RAM (read-before-write), self-checking TB |
| `code/day09_ex03_pattern_sequencer.v` | ROM-driven LED pattern player, self-checking TB |
| `code/pattern.mem` | 16-entry walking-1 LED pattern (binary format) |

## Diagrams

| File | Description |
|------|-------------|
| `diagrams/d09_mem_landscape.svg` | iCE40 memory landscape: LUT RAM vs. Block RAM vs. External |
| `diagrams/d09_sync_vs_async.svg` | Side-by-side: async read (LUTs) vs. sync read (EBR) |

## Key Concepts
- `case`-ROM vs. array + `$readmemh` ROM
- Async read → LUTs; sync read → block RAM (EBR) inference
- Single-port RAM: read-before-write vs. write-first
- iCE40 HX1K: 16 EBR × 4 Kbit = 64 Kbit total
- Counter + ROM = sequencer (LED patterns, sine waves, fonts)

## Directory Structure

```
day09_memory_ram_rom_block_ram/
├── d09_s1_rom_in_verilog.html
├── d09_s2_ram_in_verilog.html
├── d09_s3_ice40_memory_resources.html
├── d09_s4_memory_applications.html
├── code/
│   ├── day09_ex01_rom_sync.v
│   ├── day09_ex02_ram_sp.v
│   ├── day09_ex03_pattern_sequencer.v
│   └── pattern.mem
├── diagrams/
│   ├── d09_mem_landscape.svg
│   └── d09_sync_vs_async.svg
├── day09_quiz.md
└── day09_readme.md
```

---
## Code Examples

### `day09_ex01_rom_sync.v`

```verilog
// =============================================================================
// day09_ex01_rom_sync.v — Parameterized Synchronous ROM (Block RAM Inference)
// Day 9: Memory — RAM, ROM & Block RAM
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Synchronous read enables block RAM (EBR) inference on iCE40.
// One-cycle read latency: address on cycle N, data on cycle N+1.
// =============================================================================
// Build:  iverilog -DSIMULATION -o sim day09_ex01_rom_sync.v && vvp sim
// Synth:  yosys -p "read_verilog day09_ex01_rom_sync.v; synth_ice40 -top rom_sync"
// =============================================================================

module rom_sync #(
    parameter ADDR_WIDTH = 4,
    parameter DATA_WIDTH = 8,
    parameter MEM_FILE   = "pattern.mem"
)(
    input  wire                    i_clk,
    input  wire [ADDR_WIDTH-1:0]  i_addr,
    output reg  [DATA_WIDTH-1:0]  o_data
);

    reg [DATA_WIDTH-1:0] r_mem [0:(1<<ADDR_WIDTH)-1];

    initial begin
        $readmemb(MEM_FILE, r_mem);
    end

    // Synchronous read — key for block RAM inference
    always @(posedge i_clk) begin
        o_data <= r_mem[i_addr];
    end

endmodule

// =============================================================================
// Self-Checking Testbench
// =============================================================================
`ifdef SIMULATION
module tb_rom_sync;
    reg        clk = 0;
    reg  [3:0] addr;
    wire [7:0] data;

    rom_sync #(
        .ADDR_WIDTH(4), .DATA_WIDTH(8),
        .MEM_FILE("pattern.mem")
    ) uut (
        .i_clk(clk), .i_addr(addr), .o_data(data)
    );

    always #20 clk = ~clk;

    integer i, test_count = 0, fail_count = 0;

    // Expected values (must match pattern.mem)
    reg [7:0] expected [0:15];
    initial begin
        expected[ 0] = 8'b00000001;  expected[ 1] = 8'b00000010;
        expected[ 2] = 8'b00000100;  expected[ 3] = 8'b00001000;
        expected[ 4] = 8'b00010000;  expected[ 5] = 8'b00100000;
        expected[ 6] = 8'b01000000;  expected[ 7] = 8'b10000000;
        expected[ 8] = 8'b01000000;  expected[ 9] = 8'b00100000;
        expected[10] = 8'b00010000;  expected[11] = 8'b00001000;
        expected[12] = 8'b00000100;  expected[13] = 8'b00000010;
        expected[14] = 8'b00000001;  expected[15] = 8'b11111111;
    end

    initial begin
        $dumpfile("tb_rom_sync.vcd");
        $dumpvars(0, tb_rom_sync);
        addr = 0; #100;

        $display("\n=== ROM Sync Testbench ===\n");

        for (i = 0; i < 16; i = i + 1) begin
            addr = i[3:0];
            @(posedge clk);   // present address
            @(posedge clk); #1;  // data available NEXT cycle (sync read!)
            test_count = test_count + 1;
            if (data !== expected[i]) begin
                $display("FAIL: addr=%0d expected=%b got=%b", i, expected[i], data);
                fail_count = fail_count + 1;
            end else
                $display("PASS: addr=%0d = %b", i, data);
        end

        $display("\n=== SUMMARY: %0d/%0d passed ===",
                 test_count - fail_count, test_count);
        if (fail_count == 0) $display("*** ALL TESTS PASSED ***\n");
        else $display("*** %0d FAILURES ***\n", fail_count);
        $finish;
    end
endmodule
`endif
```

### `day09_ex02_ram_sp.v`

```verilog
// =============================================================================
// day09_ex02_ram_sp.v — Single-Port Synchronous RAM
// Day 9: Memory — RAM, ROM & Block RAM
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Read-before-write behavior. Synchronous read infers block RAM on iCE40.
// One-cycle read latency: address cycle N → data cycle N+1.
// =============================================================================
// Build:  iverilog -DSIMULATION -o sim day09_ex02_ram_sp.v && vvp sim
// Synth:  yosys -p "read_verilog day09_ex02_ram_sp.v; synth_ice40 -top ram_sp"
// =============================================================================

module ram_sp #(
    parameter ADDR_WIDTH = 8,
    parameter DATA_WIDTH = 8
)(
    input  wire                    i_clk,
    input  wire                    i_write_en,
    input  wire [ADDR_WIDTH-1:0]  i_addr,
    input  wire [DATA_WIDTH-1:0]  i_write_data,
    output reg  [DATA_WIDTH-1:0]  o_read_data
);

    reg [DATA_WIDTH-1:0] r_mem [0:(1<<ADDR_WIDTH)-1];

    // Read-before-write: output gets OLD value during simultaneous read/write
    always @(posedge i_clk) begin
        if (i_write_en)
            r_mem[i_addr] <= i_write_data;
        o_read_data <= r_mem[i_addr];
    end

endmodule

// =============================================================================
// Self-Checking Testbench
// =============================================================================
`ifdef SIMULATION
module tb_ram_sp;
    reg        clk = 0, we;
    reg  [7:0] addr, wdata;
    wire [7:0] rdata;

    ram_sp #(.ADDR_WIDTH(8), .DATA_WIDTH(8)) uut (
        .i_clk(clk), .i_write_en(we),
        .i_addr(addr), .i_write_data(wdata),
        .o_read_data(rdata)
    );

    always #20 clk = ~clk;

    integer i, test_count = 0, fail_count = 0;

    task check;
        input [7:0] expected;
        input [8*30-1:0] name;
    begin
        test_count = test_count + 1;
        if (rdata !== expected) begin
            $display("FAIL: %0s — expected %h, got %h", name, expected, rdata);
            fail_count = fail_count + 1;
        end else
            $display("PASS: %0s = %h", name, rdata);
    end
    endtask

    initial begin
        $dumpfile("tb_ram_sp.vcd");
        $dumpvars(0, tb_ram_sp);
        we = 0; addr = 0; wdata = 0;
        #100;

        $display("\n=== RAM Single-Port Testbench ===\n");

        // Write XOR pattern to 16 addresses
        we = 1;
        for (i = 0; i < 16; i = i + 1) begin
            addr  = i[7:0];
            wdata = i[7:0] ^ 8'hA5;
            @(posedge clk); #1;
        end
        we = 0;

        // Read back and verify (remember: 1-cycle latency!)
        for (i = 0; i < 16; i = i + 1) begin
            addr = i[7:0];
            @(posedge clk);    // present address
            @(posedge clk); #1;  // data available next cycle
            check(i[7:0] ^ 8'hA5, "read-back");
        end

        // Boundary test: address 255
        we = 1; addr = 8'hFF; wdata = 8'h42;
        @(posedge clk); #1;
        we = 0; addr = 8'hFF;
        @(posedge clk);
        @(posedge clk); #1;
        check(8'h42, "addr 255 boundary");

        $display("\n=== SUMMARY: %0d/%0d passed ===",
                 test_count - fail_count, test_count);
        if (fail_count == 0) $display("*** ALL TESTS PASSED ***\n");
        else $display("*** %0d FAILURES ***\n", fail_count);
        $finish;
    end
endmodule
`endif
```

### `day09_ex03_pattern_sequencer.v`

```verilog
// =============================================================================
// day09_ex03_pattern_sequencer.v — ROM-Based LED Pattern Player
// Day 9: Memory — RAM, ROM & Block RAM
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Steps through a ROM loaded from a .mem file.
// i_next advances the address. o_pattern drives LEDs / 7-seg.
// =============================================================================
// Build:  iverilog -DSIMULATION -o sim day09_ex03_pattern_sequencer.v && vvp sim
// Synth:  yosys -p "read_verilog day09_ex03_pattern_sequencer.v; synth_ice40 -top pattern_sequencer"
// =============================================================================

module pattern_sequencer #(
    parameter DEPTH    = 16,
    parameter WIDTH    = 8,
    parameter MEM_FILE = "pattern.mem"
)(
    input  wire             i_clk,
    input  wire             i_reset,
    input  wire             i_next,     // advance one step
    output wire [WIDTH-1:0] o_pattern
);

    reg [WIDTH-1:0] r_mem [0:DEPTH-1];

    initial begin
        $readmemb(MEM_FILE, r_mem);
    end

    reg [$clog2(DEPTH)-1:0] r_addr;

    always @(posedge i_clk) begin
        if (i_reset)
            r_addr <= 0;
        else if (i_next)
            r_addr <= (r_addr == DEPTH - 1) ? 0 : r_addr + 1;
    end

    // Async read is fine for 16 entries (LUT-based)
    assign o_pattern = r_mem[r_addr];

endmodule

// =============================================================================
// Self-Checking Testbench
// =============================================================================
`ifdef SIMULATION
module tb_pattern_sequencer;
    reg        clk = 0, reset = 1, next = 0;
    wire [7:0] pattern;

    pattern_sequencer #(
        .DEPTH(16), .WIDTH(8),
        .MEM_FILE("pattern.mem")
    ) uut (
        .i_clk(clk), .i_reset(reset),
        .i_next(next), .o_pattern(pattern)
    );

    always #20 clk = ~clk;

    integer i, test_count = 0, fail_count = 0;

    initial begin
        $dumpfile("tb_pattern_seq.vcd");
        $dumpvars(0, tb_pattern_sequencer);
        #100; reset = 0;
        @(posedge clk); #1;

        $display("\n=== Pattern Sequencer Testbench ===\n");

        // Verify first pattern at addr 0 (walking 1)
        test_count = test_count + 1;
        if (pattern !== 8'b00000001) begin
            $display("FAIL: addr 0 expected 00000001, got %b", pattern);
            fail_count = fail_count + 1;
        end else
            $display("PASS: addr 0 = %b", pattern);

        // Step through all 16 patterns
        for (i = 1; i < 16; i = i + 1) begin
            next = 1; @(posedge clk); next = 0; @(posedge clk); #1;
            test_count = test_count + 1;
            $display("  addr %0d = %b", i, pattern);
        end

        // Verify wrap-around
        next = 1; @(posedge clk); next = 0; @(posedge clk); #1;
        test_count = test_count + 1;
        if (pattern !== 8'b00000001) begin
            $display("FAIL: wrap — expected 00000001, got %b", pattern);
            fail_count = fail_count + 1;
        end else
            $display("PASS: Wrapped back to addr 0");

        $display("\n=== SUMMARY: %0d/%0d passed ===",
                 test_count - fail_count, test_count);
        if (fail_count == 0) $display("*** ALL TESTS PASSED ***\n");
        else $display("*** %0d FAILURES ***\n", fail_count);
        $finish;
    end
endmodule
`endif
```

---
## Pre-Class Self-Check Quiz

**Q1:** What is the key difference between async ROM read and sync ROM read? Which infers block RAM?

<details><summary>Answer</summary>
Asynchronous read: `assign o_data = r_mem[i_addr];` (combinational, no latency).
Synchronous read: `always @(posedge i_clk) o_data <= r_mem[i_addr];` (clocked, 1-cycle latency).
Only **synchronous read** infers block RAM (EBR).
</details>

**Q2:** How much block RAM does the iCE40 HX1K have?

<details><summary>Answer</summary>
16 Embedded Block RAM (EBR) blocks × 4 Kbit each = **64 Kbit total** (8 KB). Each block configurable as 256×16, 512×8, 1024×4, or 2048×2.
</details>

**Q3:** What system task loads a hex file into a memory array? Where does it work?

<details><summary>Answer</summary>
`$readmemh("filename.hex", r_mem)` loads hexadecimal data. `$readmemb` loads binary. Both work in **simulation** (Icarus Verilog) and **synthesis** (Yosys uses the data for initialization).
</details>

**Q4:** Write the always block for a single-port synchronous RAM with read-before-write behavior.

<details><summary>Answer</summary>

```verilog
always @(posedge i_clk) begin
    if (i_write_en)
        r_mem[i_addr] <= i_write_data;
    o_read_data <= r_mem[i_addr];  // reads old value during write
end
```
</details>